# Continual learning for second-life EV battery health: Colab demonstration

21CSC314P Big Data Essentials, Review 2. This notebook runs the project's own code from the repository on Google Colab (CPU is enough):

1. **Setup**: clone the repository, install the pinned libraries and Java (for Apache Spark).
2. **Tests**: the repository's unit tests (they use synthetic data built in temporary folders only).
3. **Big-data pipeline on real data**: download a small, checksummed slice of the public S6 dataset (Wheeler et al. 2025, CC BY 4.0, doi:10.57745/OLBXKT): the capacity checks of one cell and a few of its 100-cycle ageing files. Build the per-check labels, extract the ageing inputs with pandas and with **Apache Spark (PySpark, local mode)**, and check that the two agree.
4. **Results**: the committed result tables and figures (preprocessing, model comparison, per-group errors) and, when present, the D24 summary.

Numbers produced in step 3 come from a small subset and are a demonstration of the pipeline, not reportable results. All reported results come from the committed files in `results/`, written on clean commits (see `docs/RESULTS.md`).

## 1. Setup

The repository is private. Add a GitHub fine-grained token with read access to it as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar), and allow this notebook to use it.

In [ ]:
BRANCH = "main"   # or the branch to demonstrate
REPO = "TarunSitaraman/continual-learning-heterogeneous-second-life-ev-batteries"

import os, subprocess
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://{token}@github.com/{REPO}.git", "/content/repo"],
                   check=True)
os.chdir("/content/repo")
print(subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"], capture_output=True, text=True).stdout)

In [ ]:
%%capture install_log
!pip install -q -r requirements.txt
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless

If Colab asks to restart the session after the installation, restart it and continue from the next cell (the repository and the installed libraries stay).

In [ ]:
import glob, os, sys
os.chdir("/content/repo")   # also after a session restart
java = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk-*"))[0]
os.environ["JAVA_HOME"] = java
os.environ["PATH"] = java + "/bin:" + os.environ["PATH"]
import numpy, pandas, sklearn, torch, pyspark
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__, "| torch", torch.__version__, "| PySpark", pyspark.__version__)
!java -version 2>&1 | head -1

## 2. Tests

In [ ]:
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -3

## 3. Big-data pipeline on a real subset of S6

Downloads (all checked against the checksums committed in `docs/external_data/metadata/`):
- the authors' `extractedData.mat` (459 KB, MD5-checked), used to verify our capacity parsing;
- the capacity-check files of cell 1a (read from its zip with HTTP range requests; CRC-32-checked);
- one eighth of cell 1a's 100-cycle ageing files (`--shard 0/8`, about 4 files of roughly 1.6 million rows each).

Manifests of this partial download are written to `/content`, so the committed manifests stay unchanged.

In [ ]:
RUN_REAL_DATA_DEMO = True
CELL = "Cell1a"

import hashlib, json, urllib.request
from pathlib import Path
raw = Path("data/raw/wheeler2025"); raw.mkdir(parents=True, exist_ok=True)
record = json.loads(Path("docs/external_data/metadata/wheeler2025_lfp_dataverse.dataset.json").read_text())
files = record["data"]["latestVersion"]["files"] if "data" in record else record["latestVersion"]["files"]
mat = next(f["dataFile"] for f in files if f["dataFile"]["filename"] == "extractedData.mat")
target = raw / "extractedData.mat"
if RUN_REAL_DATA_DEMO and not target.exists():
    urllib.request.urlretrieve(f"https://entrepot.recherche.data.gouv.fr/api/access/datafile/{mat['id']}", target)
md5 = hashlib.md5(target.read_bytes()).hexdigest()
assert md5 == mat["md5"], f"MD5 mismatch: {md5} != {mat['md5']}"
print("extractedData.mat MD5 verified:", md5)

In [ ]:
if RUN_REAL_DATA_DEMO:
    !python scripts/fetch_wheeler_characterisation.py --cells {CELL} --manifest /content/charac_manifest.json
    !python scripts/build_wheeler_features.py --workers 2

In [ ]:
import pandas as pd
checks = pd.read_csv("data/wheeler_checks.csv")
print(f"{len(checks)} checks of {checks['cell'].nunique()} cell(s); {int(checks['valid'].sum())} with a C/25 label")
checks[["cell", "Battery_ID", "cycle", "check_time", "Capacity_Ah", "SOH_Percent", "valid"]].head(10)

In [ ]:
if RUN_REAL_DATA_DEMO:
    !python scripts/fetch_wheeler_characterisation.py --cells {CELL} --kind aging --shard 0/8 --manifest /content/aging_manifest.json

**Pandas reference** (`scripts/build_s6_ageing_features.py`): per-cycle responses (end-of-charge and end-of-discharge voltage, CV hold time and final current, relaxation after charge and discharge) and block aggregates (mean, last value, least-squares slope).

In [ ]:
if RUN_REAL_DATA_DEMO:
    !python scripts/build_s6_ageing_features.py --workers 2

**Apache Spark** (`scripts/spark_ageing_features.py`): one map task per ageing file parses it into per-cycle responses; the per-cycle table is cached; Spark SQL groups by (cell, block) into the same aggregates; the output is compared with the pandas reference: |Spark − pandas| ≤ 1e-9·|pandas| + 1e-12 and the same missing values.

In [ ]:
if RUN_REAL_DATA_DEMO:
    !python scripts/spark_ageing_features.py --cores 2 --tag colab
    v = json.loads(Path("results/d22_spark_validation_colab.json").read_text())
    print({k: v[k] for k in ("engine", "blocks", "cycles", "seconds", "compared_blocks", "matches_pandas")})
    print("largest |Spark - pandas|:", max(d["max_abs"] for d in v["per_input"].values()))

In [ ]:
blocks = pd.read_csv("data/s6_ageing_blocks_spark_colab.csv")
blocks.iloc[:, :8]

## 4. Committed results

From here on everything is read from the committed results files (full datasets, clean commits). The scripts regenerate the tables and figures; they do not change any number.

The demo in section 3 overwrote the committed ageing report with its subset; the next cell restores every committed results file from Git before anything is read.

In [ ]:
# The demo above rewrote results/d22_ageing_report.json for its subset; restore the committed results first.
!git checkout -- results
!python scripts/make_preprocessing_table.py > /dev/null
pd.read_csv("results/tables/preprocessing.csv")[["dataset", "stage", "value", "unit"]]

In [ ]:
!python scripts/make_d24_outputs.py
for exp in ("e1", "e2"):
    t = pd.read_csv(f"results/tables/d24_comparison_{exp}.csv")
    print(f"\nS6 stream, {'horizons 1-3' if exp == 'e1' else 'horizons 1-6'}: final average test RMSE (pp)")
    display(t[["label", "kind", "rmse_main_mean", "rmse_replicate_mean", "forgetting_main"]].round(3))

In [ ]:
from IPython.display import Image, display
for f in ("d24_comparison_e1.png", "d24_horizon_e2.png", "d24_per_group_e1.png", "spark_pipeline.png", "spark_scaling.png"):
    p = Path("paper/figures") / f
    if p.exists():
        display(Image(filename=str(p), width=760))

In [ ]:
if Path("results/forecasting_d24_e1.json").exists():
    !python scripts/summarise_d24.py
else:
    print("D24 results (RC-TFT and the literature baselines) are not committed yet.")

### Data sources and licences
- S6: W. Wheeler, Y. Bultel, P. Venet and A. Sari, "An ageing study of twenty 18650 lithium-ion Graphite/LFP cells in first and second life use," *Scientific Data* 12, 392 (2025); dataset doi:10.57745/OLBXKT, CC BY 4.0.
- S1: K. Moy *et al.*, *Data in Brief* 57, 111046 (2024); dataset doi:10.17605/OSF.IO/8JNR5 (version 2), CC BY 4.0.

Provenance, hashes and licences of every source: `docs/EXTERNAL_DATA.md`.